# Phase 1: Heterogeneous Graph Construction 
**Objective:** To construct a massive, multi-modal biomedical knowledge graph (`HeteroData`) by fusing structural knowledge from PrimeKG with polypharmacy side-effect data from the Decagon dataset.

This pipeline performs the following critical steps:
1. Aligns heterogeneous entity IDs (STITCH, DrugBank, NCBI).
2. Initializes dense biological feature vectors for drugs and diseases.
3. Compiles standard biological pathways (Protein-Protein, Disease-Phenotype, etc.).
4. Injects specific Adverse Drug Reaction (ADR) multiplex edges for downstream prediction.

In [ ]:
import time
import torch
import pandas as pd
from torch_geometric.data import HeteroData
from collections import defaultdict

print(f"Initializing Graph Construction Pipeline | PyTorch v{torch.__version__}")
start_time = time.time()

# Define core dataset paths (Updated for relative directory pathing)
PRIME_KG_FILE = '../Data/kg.csv'
COMBO_FILE = '../Data/bio-decagon-combo.csv'
MAPPING_FILE = '../Data/drug-mappings.tsv'
DRUG_FEAT_FILE = '../Data/drug_features.csv'
DISEASE_FEAT_FILE = '../Data/disease_features.csv'
ROSETTA_STONE_FILE = '../Data/nodes.csv'

# Initialize the empty PyTorch Geometric graph
data = HeteroData()

### Step 1: Entity Alignment & Dictionary Mapping
Biomedical datasets use disparate ID systems. We must create a unified "Rosetta Stone" mapping STITCH IDs (from the polypharmacy dataset) to DrugBank IDs (from PrimeKG), while mapping all unique string IDs to zero-indexed integers required by PyTorch.

In [ ]:
print("Aligning STITCH-to-DrugBank nomenclatures...")
mapping_df = pd.read_csv(MAPPING_FILE, sep='\t').dropna(subset=['drugbankId', 'stitch_id'])
stitch_to_drugbank_map = pd.Series(
    mapping_df.drugbankId.str.strip().values, 
    index=mapping_df.stitch_id.str.strip()
).to_dict()

print("Indexing global PrimeKG topology...")
primekg_df = pd.read_csv(PRIME_KG_FILE, low_memory=False)
primekg_df['x_id'] = primekg_df['x_id'].astype(str).str.strip()
primekg_df['y_id'] = primekg_df['y_id'].astype(str).str.strip()

all_nodes = pd.concat([
    primekg_df[['x_id', 'x_type']].rename(columns={'x_id': 'id', 'x_type': 'type'}),
    primekg_df[['y_id', 'y_type']].rename(columns={'y_id': 'id', 'y_type': 'type'})
]).drop_duplicates(subset='id')

node_to_int_map = {}
for node_type in all_nodes['type'].unique():
    nodes_of_type = all_nodes[all_nodes['type'] == node_type]['id'].values
    node_to_int_map[node_type] = {node_id: i for i, node_id in enumerate(nodes_of_type)}

print("Loading localized node indices...")
nodes_df = pd.read_csv(ROSETTA_STONE_FILE, low_memory=False)
index_to_string_id_map = pd.Series(
    nodes_df.node_id.astype(str).str.strip().values, 
    index=nodes_df.node_index
).to_dict()

print(f"Successfully mapped {len(all_nodes)} global entities.")

### Step 2: Node Feature Injection
We inject pre-computed biological and chemical features for explicit node types (`drug` and `disease`). For auxiliary nodes (e.g., proteins, phenotypes), we initialize strictly orthogonal identity matrices to allow the GNN to learn structural embeddings from scratch.

In [ ]:
print("Injecting dense feature vectors...")

def load_node_features(file_path, node_type, id_cleaner=lambda x: x):
    df = pd.read_csv(file_path, sep=',', header=0, on_bad_lines='skip', engine='python')
    feature_cols = [col for col in df.columns if col not in ['node_index', 'node_id']]
    
    num_nodes = len(node_to_int_map.get(node_type, {}))
    feature_tensor = torch.zeros(num_nodes, len(feature_cols), dtype=torch.float32)
    
    loaded_count = 0
    for row in df.itertuples(index=False):
        string_id = index_to_string_id_map.get(row.node_index)
        if string_id:
            cleaned_id = id_cleaner(string_id)
            if cleaned_id in node_to_int_map.get(node_type, {}):
                int_id = node_to_int_map[node_type][cleaned_id]
                features = pd.to_numeric([getattr(row, col) for col in feature_cols], errors='coerce')
                feature_tensor[int_id] = torch.tensor(features, dtype=torch.float32).nan_to_num(0.0)
                loaded_count += 1
                
    data[node_type].x = feature_tensor
    print(f" -> Allocated {loaded_count} features for '{node_type}'.")

# Load complex features
load_node_features(DRUG_FEAT_FILE, 'drug', lambda x: x.replace("DrugBank::", "").strip())
load_node_features(DISEASE_FEAT_FILE, 'disease')

# Allocate lightweight structural latent vectors for auxiliary nodes
print("Allocating 128-dim latent vectors for auxiliary nodes...")
FEATURE_DIM = 128
for node_type in all_nodes['type'].unique():
    if node_type not in ['drug', 'disease']:
        num_nodes = len(node_to_int_map.get(node_type, {}))
        if num_nodes > 0:
            # INSTANT COMPRESSION: Skip the massive identity matrix completely!
            data[node_type].x = torch.randn((num_nodes, FEATURE_DIM), dtype=torch.float32)

### Step 3: Base Topology & Polypharmacy Edge Generation
We construct the core message-passing topology by extracting established biological pathways from PrimeKG. Subsequently, we overlay the multiplex Adverse Drug Reaction (ADR) target edges derived from polypharmacy combinations.

In [ ]:
print("Extracting foundational knowledge graph topology...")
edge_data = defaultdict(lambda: [[], []]) 

for row in primekg_df.itertuples(index=False):
    src_type, dst_type = row.x_type, row.y_type
    src_id, dst_id = row.x_id, row.y_id 
    rel_type = str(row.relation).replace(" ", "_")
    
    if (src_type in node_to_int_map and dst_type in node_to_int_map and 
        src_id in node_to_int_map[src_type] and dst_id in node_to_int_map[dst_type]):
        
        edge_type_tuple = (src_type, rel_type, dst_type)
        edge_data[edge_type_tuple][0].append(node_to_int_map[src_type][src_id])
        edge_data[edge_type_tuple][1].append(node_to_int_map[dst_type][dst_id])

for edge_type_tuple, (src_list, dst_list) in edge_data.items():
    if src_list:
        data[edge_type_tuple].edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)

print("Overlaying polypharmacy ADR multiplex edges...")
combo_df = pd.read_csv(COMBO_FILE)
combo_df['drug_1_id'] = combo_df['STITCH 1'].astype(str).str.strip().map(stitch_to_drugbank_map)
combo_df['drug_2_id'] = combo_df['STITCH 2'].astype(str).str.strip().map(stitch_to_drugbank_map)
mappable_combo_df = combo_df.dropna(subset=['drug_1_id', 'drug_2_id'])

adr_edge_data = defaultdict(lambda: [[], []])
drug_map = node_to_int_map.get('drug', {})

mappable_combo_df.columns = mappable_combo_df.columns.str.replace(' ', '_')

for row in mappable_combo_df.itertuples(index=False):
    if row.drug_1_id in drug_map and row.drug_2_id in drug_map:
        clean_adr = f"causes_{''.join(e for e in str(row.Side_Effect_Name) if e.isalnum() or e=='_')}" 
        edge_type = ('drug', clean_adr, 'drug')
        adr_edge_data[edge_type][0].append(drug_map[row.drug_1_id])
        adr_edge_data[edge_type][1].append(drug_map[row.drug_2_id])

for edge_type, (src, dst) in adr_edge_data.items():
    if src: 
        data[edge_type].edge_index = torch.tensor([src, dst], dtype=torch.long)

print("Topology generation complete.")

### Step 4: Object Validation & Export
Validating the structural integrity of the `HeteroData` object before serializing it to disk for the HPC training cluster.

In [ ]:
import os
import torch
from torch_geometric.data import HeteroData

print("Commencing Surgical Graph Validation and Export...")

# 1. Create a pristine, mathematically pure clone
clean_data = HeteroData()

# 2. Extract tensor node features and explicitly set num_nodes
for node_type in data.node_types:
    if 'x' in data[node_type]:
        clean_x = data[node_type].x.to(torch.float32).contiguous()
        clean_data[node_type].x = clean_x
        clean_data[node_type].num_nodes = clean_x.size(0)

# 3. Extract tensor edge indices
for edge_type in data.edge_types:
    edge_tensor = data[edge_type].edge_index.to(torch.long).contiguous()
    clean_data[edge_type].edge_index = edge_tensor
    
    # Failsafe: Infer num_nodes from edges if features are missing
    src_type, _, dst_type = edge_type
    if 'num_nodes' not in clean_data[src_type]:
        clean_data[src_type].num_nodes = int(edge_tensor[0].max()) + 1
    if 'num_nodes' not in clean_data[dst_type]:
        clean_data[dst_type].num_nodes = int(edge_tensor[1].max()) + 1

# 4. Save the optimized graph
save_path = '../Data/MTP_Graph.pt'
torch.save(clean_data, save_path)

file_size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"✅ Success! Graph compressed and serialized.")
print(f"📊 Final Publication-Ready File Size: {file_size_mb:.2f} MB")
print(f"🧠 Topology: {clean_data.num_nodes} Nodes | {clean_data.num_edges} Edges")